In [ ]:
import torch

# Check if CUDA (GPU) is available
if torch.cuda.is_available():
    print(f"✅ CUDA is active. GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("❌ CUDA not detected. To enable: Go to Runtime > Change runtime type > Select T4 GPU.")
    DEVICE = "cpu"

✅ CUDA is active. GPU: Tesla T4


In [ ]:
import warnings
import datetime
# Suppress specific deprecation warnings to clean up the output
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")

In [ ]:
%pip install --upgrade stable-baselines3 gymnasium

#Major Project I: Review 1 Baseline Implementations
This notebook contains the foundational baseline models for Review 1. It is divided into two main components: the Reinforcement Learning environment wiring (PPO) and the Traffic Speed Prediction baseline (LSTM).

#Part 1: LSTM Baseline for Traffic Speed Prediction
Objective: Establish a baseline single-sensor speed prediction model using the METR-LA traffic dataset. This transforms raw data into windowed sequences, trains an LSTM, and plots predicted versus actual speeds.*italicized text*

###1.1 Imports and Configuration

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("annnnguyen/metr-la-dataset")

print("Path to dataset files:", path)

100%|██████████| 12.5M/12.5M [00:01<00:00, 7.31MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/annnnguyen/metr-la-dataset/versions/4


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
SENSOR_ID = 0          # pick 1 sensor col for baseline demo, scale to all 207 later
SEQ_LEN = 12           # 12 steps * 5min = 1hr history window
PRED_LEN = 1           # predict next 5min step
BATCH_SIZE = 32
EPOCHS = 15
HIDDEN_SIZE = 64
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Overriding DEVICE based on the global check at the top
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Part 1 Training will use: {DEVICE}")

Part 1 Training will use: cuda


###1.2 Data Loading and Preprocessing

In [ ]:
import os
from torch.utils.data import Dataset, DataLoader

# ---------------- LOAD DATA ----------------
# METR-LA standard file: metr-la.h5
def load_metr_la(path_root=None):
    # If path_root is provided (from kagglehub), use it; otherwise default to local
    if path_root:
        file_path = os.path.join(path_root, 'METR-LA.h5')
    else:
        file_path = 'metr-la.h5'

    print(f"Loading data from: {file_path}")
    df = pd.read_hdf(file_path)
    return df

# ---------------- PREPROCESS ----------------
def preprocess(df, sensor_col):
    series = df.iloc[:, sensor_col].values.astype(np.float32)

    # z-score normalize
    mean, std = series.mean(), series.std()
    series_norm = (series - mean) / std

    return series_norm, mean, std

def make_windows(series, seq_len, pred_len):
    X, y = [], []
    for i in range(len(series) - seq_len - pred_len + 1):
        X.append(series[i : i + seq_len])
        y.append(series[i + seq_len : i + seq_len + pred_len])
    return np.array(X), np.array(y)

class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # (N, seq_len, 1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

###1.3 LSTM Model Architecture

In [ ]:
import torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj):
        # x: [batch_size, num_nodes, in_features]
        # support: [batch_size, num_nodes, out_features]
        support = torch.matmul(x, self.weight)
        # output: [batch_size, num_nodes, out_features]
        output = torch.matmul(adj, support)
        return F.relu(output)

class TGCN_LSTM(nn.Module):
    def __init__(self, num_nodes, in_features, gcn_out, lstm_hidden, out_horizon):
        super(TGCN_LSTM, self).__init__()
        self.gcn = GCNLayer(in_features, gcn_out)
        self.lstm = nn.LSTM(input_size=gcn_out, hidden_size=lstm_hidden, batch_first=True)
        self.linear = nn.Linear(lstm_hidden, out_horizon)

    def forward(self, x, adj):
        # x: [batch_size, seq_len, num_nodes, in_features]
        batch_size, seq_len, num_nodes, _ = x.size()

        gcn_outputs = []
        for t in range(seq_len):
            x_t = x[:, t, :, :]
            g_out = self.gcn(x_t, adj)
            gcn_outputs.append(g_out.unsqueeze(1))

        x_spatial = torch.cat(gcn_outputs, dim=1)
        x_spatial = x_spatial.view(batch_size * num_nodes, seq_len, -1)

        lstm_out, _ = self.lstm(x_spatial)
        last_hidden = lstm_out[:, -1, :]

        out = self.linear(last_hidden)
        return out.view(batch_size, num_nodes, -1)

### 1.5 Adjacency Matrix Preparation
TGCN requires a normalized adjacency matrix. We load the sensor distances and apply a Gaussian kernel.

In [ ]:
import pickle

def get_adjacency_matrix(path_root, num_nodes=207):
    # Standard METR-LA adj_mx path
    adj_path = os.path.join(path_root, 'adj_mx.pkl')
    with open(adj_path, 'rb') as f:
        _, _, adj_mx = pickle.load(f, encoding='latin1')

    # Convert to tensor and move to device
    adj = torch.tensor(adj_mx, dtype=torch.float32).to(DEVICE)
    return adj

# Example usage for initialization
# ADJ = get_adjacency_matrix(path)

### 1.6 TGCN Training Loop
This loop trains the model on the full graph (all sensors) simultaneously.

In [ ]:
import pickle
import torch

def load_adj_mx(path_root, sigma2=0.1, epsilon=0.1):
    """
    Loads METR-LA adjacency matrix and applies Gaussian kernel normalization.

    Args:
        path_root: Directory containing adj_mx.pkl
        sigma2: Standard deviation for the Gaussian kernel
        epsilon: Threshold for sparsity (weights below this are set to 0)
    """
    adj_path = os.path.join(path_root, 'adj_mx.pkl')

    with open(adj_path, 'rb') as f:
        # METR-LA pkl usually returns (sensor_ids, sensor_id_to_ind, adj_mx)
        sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')

    # Normalize using Gaussian Kernel (standard for STGCN/TGCN)
    distances = adj_mx[~np.isinf(adj_mx)]
    std = distances.std()
    adj_mx = np.exp(-np.square(adj_mx / std))

    # Apply threshold to keep the graph sparse
    adj_mx[adj_mx < epsilon] = 0

    # Convert to Tensor
    adj_tensor = torch.tensor(adj_mx, dtype=torch.float32).to(DEVICE)

    print(f"Loaded adjacency matrix for {len(sensor_ids)} sensors.")
    print(f"Matrix shape: {adj_tensor.shape}")

    return sensor_ids, sensor_id_to_ind, adj_tensor

# Usage example:
# sensor_ids, sensor_map, ADJ_MAT = load_adj_mx(path)

# Part 2: PPO Agent and Stop Sequencing Environment
Objective: Prove the wiring works between the Gymnasium environment, Stable Baselines 3 (PPO), and the reward mechanics. This utilizes a toy environment where distances will later be replaced by real Amazon dataset stops and LSTM travel-time signals.

###2.1 download amazon last mile dataset and dependencies

In [ ]:
%pip install --upgrade stable-baselines3 gymnasium

In [ ]:
# 1. Install the AWS CLI package
!pip install awscli

# 2. Verify the installation works (should print the version number)
!aws --version

aws-cli/1.45.44 Python/3.12.13 Linux/6.6.122+ botocore/1.43.44


In [ ]:
# 1. Create a directory for the dataset
!mkdir -p /content/amazon_last_mile

# 2. Sync the public AWS S3 bucket to your Colab environment
# This will pull down the route_data, package_data, actual_sequences, and travel_times JSONs
print("Downloading Official Amazon Last Mile dataset from AWS...")
!aws s3 sync s3://amazon-last-mile-challenges/ /content/amazon_last_mile/ --no-sign-request

print("Download complete! Files saved in /content/amazon_last_mile/")

# 3. List the downloaded files to verify
!ls -lh /content/amazon_last_mile/

Download complete! Files saved in /content/amazon_last_mile/
total 4.0K
drwxr-xr-x 4 root root 4.0K Jul  9 08:28 almrrc2021


In [ ]:
import json
import gc # Garbage collector to free up RAM

ROUTE_FILE = '/content/amazon_last_mile/almrrc2021/almrrc2021-data-training/model_build_inputs/route_data.json'
PACKAGE_FILE = '/content/amazon_last_mile/almrrc2021/almrrc2021-data-training/model_build_inputs/package_data.json'
TRAVEL_FILE = '/content/amazon_last_mile/almrrc2021/almrrc2021-data-training/model_build_inputs/travel_times.json'
# ---------------------------------------------------------
# STEP 1: Filter Route Data for Los Angeles ('DLA' stations)
# ---------------------------------------------------------
print("Loading massive route dataset...")
with open(ROUTE_FILE, 'r') as f:
    all_routes = json.load(f)

la_routes = {}
for route_id, route_info in all_routes.items():
    if route_info['station_code'].startswith('DLA'):
        la_routes[route_id] = route_info

print(f"Total routes worldwide: {len(all_routes)}")
print(f"Los Angeles routes found: {len(la_routes)}")

# Free up memory immediately
del all_routes
gc.collect()

# Save our valid LA IDs to filter the next files
valid_la_route_ids = set(la_routes.keys())

# ---------------------------------------------------------
# STEP 2: Filter Package Data (Only keep LA packages)
# ---------------------------------------------------------
print("Loading and filtering package data...")
with open(PACKAGE_FILE, 'r') as f:
    all_packages = json.load(f)

la_packages = {r_id: all_packages[r_id] for r_id in valid_la_route_ids if r_id in all_packages}

del all_packages
gc.collect()

# ---------------------------------------------------------
# STEP 3: Filter Travel Times (Distance Matrix)
# ---------------------------------------------------------
print("Loading and filtering travel times (This takes a moment)...")
with open(TRAVEL_FILE, 'r') as f:
    all_travel_times = json.load(f)

la_travel_times = {r_id: all_travel_times[r_id] for r_id in valid_la_route_ids if r_id in all_travel_times}

del all_travel_times
gc.collect()

print("✅ Success! LA data isolated and RAM cleared.")

Loading massive route dataset...
Total routes worldwide: 6112
Los Angeles routes found: 2888
Loading and filtering package data...
Loading and filtering travel times (This takes a moment)...
✅ Success! LA data isolated and RAM cleared.


###2.2 Imports and Configuration

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# Double-check versions if needed
import stable_baselines3
print(f"SB3 Version: {stable_baselines3.__version__}")

N_STOPS = 10  # toy: 10 delivery stops

SB3 Version: 2.9.0


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 2.3 Environment Definition

In [ ]:
class StopSequencingEnv(gym.Env):
    """
    State  = which stops visited (binary mask) + current position (index)
    Action = pick next stop (discrete, N_STOPS choices)
    Reward = -distance traveled (minimize total route distance)
    Episode ends when all stops visited.
    """

    def __init__(self, n_stops=N_STOPS, seed=42):
        super().__init__()
        self.n_stops = n_stops
        rng = np.random.default_rng(seed)
        # toy coordinates, replace with real lat/lon from Amazon dataset later
        self.coords = rng.uniform(0, 100, size=(n_stops, 2))

        self.action_space = spaces.Discrete(n_stops)
        # obs = visited_mask (n_stops) + current_pos_onehot (n_stops)
        self.observation_space = spaces.Box(low=0, high=1, shape=(n_stops * 2,), dtype=np.float32)

        self.reset()

    def _get_obs(self):
        pos_onehot = np.zeros(self.n_stops, dtype=np.float32)
        pos_onehot[self.current] = 1.0
        return np.concatenate([self.visited.astype(np.float32), pos_onehot])

    def _dist(self, a, b):
        return np.linalg.norm(self.coords[a] - self.coords[b])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current = 0
        self.visited = np.zeros(self.n_stops, dtype=bool)
        self.visited[0] = True
        self.steps = 0
        return self._get_obs(), {}

    def step(self, action):
        self.steps += 1

        if self.visited[action]:
            # picked already-visited stop -> penalty, no move
            reward = -10.0
            terminated = False
        else:
            dist = self._dist(self.current, action)
            reward = -dist                       # minimize travel distance
            self.current = action
            self.visited[action] = True
            terminated = bool(self.visited.all())

        truncated = self.steps >= self.n_stops * 2   # safety cap
        return self._get_obs(), reward, terminated, truncated, {}

    # NOTE: real integration point ->
    # replace self._dist() with: LSTM predicted travel time between stops
    # i.e. state includes lstm_predicted_time(current, candidate) not raw euclidean dist

### 2.4 Training and Baseline Evaluation Demonstration

In [ ]:
def train_demo():
    env = StopSequencingEnv()
    check_env(env)  # validates Gym API compliance -- show this passes to panel

    model = PPO("MlpPolicy", env, verbose=1, n_steps=256, batch_size=64)
    model.learn(total_timesteps=20_000)

    model.save("ppo_stop_sequencing_demo")

    # quick eval -- show return improves over random policy
    obs, _ = env.reset()
    total_reward = 0
    for _ in range(N_STOPS * 2):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

    print(f"Trained agent route reward: {total_reward:.2f}")

    # baseline: random policy for comparison
    obs, _ = env.reset()
    random_reward = 0
    for _ in range(N_STOPS * 2):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)
        random_reward += reward
        if terminated or truncated:
            break
    print(f"Random baseline route reward: {random_reward:.2f}")

    return model

# Execute the demo
train_demo()

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 19.2     |
|    ep_rew_mean     | -530     |
| time/              |          |
|    fps             | 223      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 256      |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 19.5         |
|    ep_rew_mean          | -534         |
| time/                   |              |
|    fps                  | 228          |
|    iterations           | 2            |
|    time_elapsed         | 2            |
|    total_timesteps      | 512          |
| train/                  |              |
|    approx_kl            | 0.0013241102 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.3         |
|    explained_variance   | -0.00202     |
|    learning_r

# Part 3: OSMnx Real-Road Network + Live PPO Simulation
Toy env above = euclidean distance, fake coords. This part swaps in real LA streets.
Stops = real graph nodes. Reward = real travel_time (not straight-line).
Output = live animated map, PPO route playback on actual roads.

In [ ]:
!pip install --upgrade osmnx folium scikit-learn

In [ ]:
import sys
# Force install NumPy 2.0.0 and scikit-learn to fix the 'numpy.strings' error
!{sys.executable} -m pip install --upgrade "numpy>=2.0.0" scikit-learn --quiet

# Verification
import numpy as np
import sklearn
print(f"Numpy version: {np.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")

Numpy version: 2.5.1
scikit-learn version: 1.9.0


### 3.1 Download real road graph (Downtown LA)

In [ ]:
import osmnx as ox
import networkx as nx
import folium
import numpy as np
import datetime as dt
import sklearn
# Downtown LA center point -- matches Amazon 'DLA' station coverage area
CENTER_LAT, CENTER_LON = 34.0407, -118.2468
GRAPH_DIST_M = 3000  # radius in meters, keep small = fast download + fast Dijkstra

print("Downloading OSM drive-network graph...")
G = ox.graph_from_point((CENTER_LAT, CENTER_LON), dist=GRAPH_DIST_M, network_type="drive")
G = ox.add_edge_speeds(G)        # fills speed_kph per edge
G = ox.add_edge_travel_times(G)  # fills travel_time (sec) per edge
print(f"Nodes: {len(G.nodes)}  Edges: {len(G.edges)}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Nodes: 2342  Edges: 6307


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 3.2 Pick real delivery stops (graph nodes, not fake coords)

In [ ]:
# REPLACE CELL 34
TARGET_ROUTE_ID = next(iter(la_routes))  # swap manually for specific route
route_stops = la_routes[TARGET_ROUTE_ID]['stops']

stop_ids  = list(route_stops.keys())
stop_lats = [route_stops[s]['lat'] for s in stop_ids]
stop_lons = [route_stops[s]['lng'] for s in stop_ids]

N_STOPS = len(stop_ids)
print(f"Route {TARGET_ROUTE_ID} -> {N_STOPS} stops = ALL packages, route")

stop_nodes = ox.distance.nearest_nodes(G, X=stop_lons, Y=stop_lats)
stop_coords = np.array([[G.nodes[n]['y'], G.nodes[n]['x']] for n in stop_nodes])

# sanity check -- graph radius 3000m may not cover full route
snap_dist = [np.hypot(stop_coords[i][0]-stop_lats[i], stop_coords[i][1]-stop_lons[i]) for i in range(N_STOPS)]
bad = [i for i,d in enumerate(snap_dist) if d > 0.01]
if bad:
    print(f"WARNING: {len(bad)} stops snapped far off graph. Bump GRAPH_DIST_M in cell 32.")

Route RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77 -> 119 stops = ALL packages, route


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Retry the snapping operation
stop_nodes = ox.distance.nearest_nodes(G, X=stop_lons, Y=stop_lats)
stop_coords = np.array([[G.nodes[n]['y'], G.nodes[n]['x']] for n in stop_nodes])
print(f"Successfully snapped {len(stop_nodes)} nodes to the graph.")

Successfully snapped 119 nodes to the graph.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### 3.3 Real distance matrix -- shortest-path travel_time, not straight line

In [ ]:
def build_distance_matrix(G, nodes, weight="travel_time"):
    n = len(nodes)
    dist = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        lengths = nx.single_source_dijkstra_path_length(G, nodes[i], weight=weight)
        for j in range(n):
            dist[i, j] = lengths.get(nodes[j], 1e6)  # unreachable pair -> big penalty
    return dist

DIST_MATRIX = build_distance_matrix(G, stop_nodes, weight="travel_time")
print("Distance matrix (sec travel time):")
print(DIST_MATRIX)

Distance matrix (sec travel time):
[[ 0.        0.        0.       ... 35.730316  0.        0.      ]
 [ 0.        0.        0.       ... 35.730316  0.        0.      ]
 [ 0.        0.        0.       ... 35.730316  0.        0.      ]
 ...
 [35.730316 35.730316 35.730316 ...  0.       35.730316 35.730316]
 [ 0.        0.        0.       ... 35.730316  0.        0.      ]
 [ 0.        0.        0.       ... 35.730316  0.        0.      ]]


### 3.4 Baseline Model Evaluation (Nearest Neighbor & OR-Tools)

### 3.4 Env swap -- same MDP, real reward
Reuse `StopSequencingEnv` class from Part 2. Only `_dist()` changes:
euclidean -> real road travel_time lookup.

In [ ]:
class OSMStopSequencingEnv(StopSequencingEnv):
    """Same state/action/reward structure as StopSequencingEnv.
    _dist() now pulls real road travel_time instead of euclidean toy distance."""

    def __init__(self, dist_matrix, seed=42):
        self.dist_matrix = dist_matrix
        super().__init__(n_stops=dist_matrix.shape[0], seed=seed)

    def _dist(self, a, b):
        # Explicitly cast to float to satisfy Gymnasium/SB3 check_env requirements
        return float(self.dist_matrix[a, b])

env = OSMStopSequencingEnv(DIST_MATRIX)
check_env(env)
print("Env OK -- real-road reward wired in.")

Env OK -- real-road reward wired in.


In [ ]:
import numpy as np

def get_action_mask(env):
    # 1 for valid (unvisited), 0 for invalid (visited)
    mask = (~env.visited).astype(np.int8)
    return mask

# Diagnostic check of the current observation
obs, _ = env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Visited Mask (first half): {obs[:N_STOPS]}")
print(f"Current Pos (second half): {obs[N_STOPS:]}")

Observation shape: (238,)
Visited Mask (first half): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Current Pos (second half): [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [ ]:
def get_route_path(model, env):
    """Helper to extract the stop sequence chosen by the PPO model."""
    obs, _ = env.reset()
    route_idx = [env.current]
    visited_set = {int(env.current)}

    for _ in range(env.n_stops * 2):
        action, _ = model.predict(obs, deterministic=True)
        action = int(action)

        # Heuristic: if model repeats a visit, pick the nearest unvisited stop
        if action in visited_set:
            unvisited = [idx for idx in range(env.n_stops) if idx not in visited_set]
            if not unvisited: break
            action = int(min(unvisited, key=lambda x: env.dist_matrix[env.current, x]))

        obs, reward, terminated, truncated, _ = env.step(action)
        route_idx.append(action)
        visited_set.add(action)

        if terminated or truncated: break

    return route_idx

### 3.5 Train PPO on real-road env

In [ ]:
osm_model = PPO("MlpPolicy", env, verbose=1, n_steps=256, batch_size=64)
osm_model.learn(total_timesteps=20_000)
osm_model.save("ppo_osm_stop_sequencing")

# Quick evaluation vs random
obs, _ = env.reset()
trained_reward = 0
for _ in range(N_STOPS * 2):
    action, _ = osm_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    trained_reward += reward
    if terminated or truncated:
        break
print(f"Trained agent total travel_time reward: {trained_reward:.2f}")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 238       |
|    ep_rew_mean     | -2.14e+03 |
| time/              |           |
|    fps             | 517       |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 256       |
----------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 238           |
|    ep_rew_mean          | -2.78e+03     |
| time/                   |               |
|    fps                  | 246           |
|    iterations           | 2             |
|    time_elapsed         | 2             |
|    total_timesteps      | 512           |
| train/                  |               |
|    approx_kl            | 0.00024694693 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -4.78         |
|    explained_variance   | -0.000

### 3.6 Extract stop order PPO picked

In [ ]:
def get_route_path_detailed(model, env):
    obs, _ = env.reset()
    route_idx = [env.current]
    visited_set = {int(env.current)}
    total_actual_time = 0

    print(f"{'Step':<5} | {'From':<8} | {'To':<8} | {'Leg Travel Time (s)':<20}")
    print("-" * 55)

    for i in range(env.n_stops * 3):
        raw_action, _ = model.predict(obs, deterministic=True)
        action = int(raw_action)

        # Force unique visit heuristic if model repeats
        if action in visited_set:
            unvisited = [idx for idx in range(env.n_stops) if idx not in visited_set]
            if not unvisited: break
            action = int(min(unvisited, key=lambda x: env.dist_matrix[env.current, x]))

        leg_time = float(env.dist_matrix[env.current, action])
        total_actual_time += leg_time

        print(f"{i+1:<5} | Stop {env.current:<3} | Stop {action:<3} | {leg_time:<20.2f}")

        obs, reward, terminated, truncated, _ = env.step(action)
        route_idx.append(action)
        visited_set.add(action)

        if terminated or truncated: break

    print("-" * 55)
    print(f"TOTAL ROUTE TRAVEL TIME: {total_actual_time:.2f} seconds")
    return route_idx

# Re-run evaluation
route_idx = get_route_path_detailed(osm_model, env)
route_node_seq = [stop_nodes[i] for i in route_idx]
print("\nFinal Route Sequence:", route_idx)

Step  | From     | To       | Leg Travel Time (s) 
-------------------------------------------------------
1     | Stop 0   | Stop 79  | 0.00                
2     | Stop 79  | Stop 1   | 0.00                
3     | Stop 1   | Stop 2   | 0.00                
4     | Stop 2   | Stop 4   | 0.00                
5     | Stop 4   | Stop 5   | 0.00                
6     | Stop 5   | Stop 6   | 0.00                
7     | Stop 6   | Stop 7   | 0.00                
8     | Stop 7   | Stop 8   | 0.00                
9     | Stop 8   | Stop 9   | 0.00                
10    | Stop 9   | Stop 10  | 0.00                
11    | Stop 10  | Stop 12  | 0.00                
12    | Stop 12  | Stop 13  | 0.00                
13    | Stop 13  | Stop 14  | 0.00                
14    | Stop 14  | Stop 15  | 0.00                
15    | Stop 15  | Stop 16  | 0.00                
16    | Stop 16  | Stop 18  | 0.00                
17    | Stop 18  | Stop 19  | 0.00                
18    | Stop 19  | Stop 21

### 3.7 Live simulation on real map
Draws actual road-following polyline (not straight lines) between each stop pair,
using real shortest path per leg. TimestampedGeoJson = play/pause timeline slider,
route draws itself leg by leg -- this is the "live" part.

In [ ]:
from folium.plugins import TimestampedGeoJson

m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=14, tiles="cartodbpositron")

features = []
base_time = dt.datetime(2026, 1, 1, 8, 0, 0)
t_cursor = base_time
total_seconds_travelled = 0

arrival_times = {route_idx[0]: base_time}

for step, (a, b) in enumerate(zip(route_node_seq[:-1], route_node_seq[1:])):
    try:
        path = nx.shortest_path(G, a, b, weight="travel_time")
        leg_time = nx.shortest_path_length(G, a, b, weight="travel_time")
    except nx.NetworkXNoPath:
        leg_time = 0
        continue

    coords = [[G.nodes[n]['x'], G.nodes[n]['y']] for n in path]
    n_pts = len(coords)

    features.append({
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": coords},
        "properties": {"times": [t_cursor.isoformat()] * n_pts,
                        "style": {"color": "orange", "weight": 4}},
    })

    step_secs = max(leg_time, 1) / max(n_pts - 1, 1)
    for k, c in enumerate(coords):
        t_pt = (t_cursor + dt.timedelta(seconds=k * step_secs)).isoformat()
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": c},
            "properties": {"times": [t_pt], "icon": "circle",
                            "iconstyle": {"fillColor": "black", "fillOpacity": 1,
                                          "stroke": "false", "radius": 4}},
        })

    t_cursor += dt.timedelta(seconds=max(leg_time, 1))
    total_seconds_travelled += leg_time
    target_stop_idx = route_idx[step + 1]
    arrival_times[target_stop_idx] = t_cursor

TimestampedGeoJson({"type": "FeatureCollection", "features": features},
    period="PT30S", add_last_point=True, transition_time=200, loop=False).add_to(m)

for order, stop_i in enumerate(route_idx):
    n = stop_nodes[stop_i]
    is_start = (order == 0)
    color = "#d9534f" if is_start else "#337ab7"
    arrival_str = arrival_times.get(stop_i, base_time).strftime('%H:%M:%S')
    label = f"Order: {order} | Stop: {stop_i}<br>ETA: {arrival_str}"
    icon_html = f'''<div style="background-color:{color};color:white;border:2px solid white;border-radius:50%;width:28px;height:28px;text-align:center;line-height:24px;font-weight:bold;font-size:10px;box-shadow:0 0 5px rgba(0,0,0,0.2);">{order}</div>'''
    folium.Marker([G.nodes[n]['y'], G.nodes[n]['x']], popup=folium.Popup(label, max_width=200), tooltip=f"Stop {stop_i}",
                  icon=folium.DivIcon(html=icon_html, icon_size=(28,28), icon_anchor=(14,14))).add_to(m)

# Print total time taken
hours = int(total_seconds_travelled // 3600)
minutes = int((total_seconds_travelled % 3600) // 60)
print(f"--- ROUTE STATISTICS ---")
print(f"Total Travel Time: {total_seconds_travelled:.2f} seconds")
print(f"Formatted Time: {hours}h {minutes}m")
print(f"------------------------")

display(m)

--- ROUTE STATISTICS ---
Total Travel Time: 582.07 seconds
Formatted Time: 0h 9m
------------------------


In [ ]:
!pip install ortools

### Logic: How to generate road-following legs
After the PPO model provides a sequence of stop indices (e.g., `[0, 8, 1, 9...]`), we convert these to real OpenStreetMap node IDs. To get the actual route geometry, we perform the following for every step:
1. **Identify the Leg**: Take node $A$ (current) and node $B$ (next).
2. **Pathfinding**: Use Dijkstra's algorithm via `nx.shortest_path(G, A, B, weight='travel_time')` to find the exact sequence of intersections/nodes between them.
3. **Coordinate Extraction**: Map those nodes back to their Lat/Lon coordinates for mapping.

In [ ]:
import networkx as nx

# Using the actual route sequence generated by our PPO model (route_idx)
print(f"Processing first few stops of the PPO sequence: {route_idx[:4]}\n")

for i in range(len(route_idx[:3])):
    start_node = stop_nodes[route_idx[i]]
    end_node = stop_nodes[route_idx[i+1]]

    try:
        # Calculate the road-following path for this specific leg
        leg_path = nx.shortest_path(G, start_node, end_node, weight='travel_time')
        leg_time = nx.shortest_path_length(G, start_node, end_node, weight='travel_time')

        print(f"Leg {i+1}: From Stop {route_idx[i]} to Stop {route_idx[i+1]}")
        print(f"  - Road distance involves {len(leg_path)} network nodes")
        print(f"  - Estimated travel time: {leg_time:.2f} seconds")
        print(f"  - Path start coords: ({G.nodes[leg_path[0]]['y']}, {G.nodes[leg_path[0]]['x']})\n")
    except nx.NetworkXNoPath:
        print(f"Leg {i+1}: No valid road path found between nodes.")

Processing first few stops of the PPO sequence: [0, 79, 1, 2]

Leg 1: From Stop 0 to Stop 79
  - Road distance involves 1 network nodes
  - Estimated travel time: 0.00 seconds
  - Path start coords: (34.0674019, -118.2753526)

Leg 2: From Stop 79 to Stop 1
  - Road distance involves 1 network nodes
  - Estimated travel time: 0.00 seconds
  - Path start coords: (34.0674019, -118.2753526)

Leg 3: From Stop 1 to Stop 2
  - Road distance involves 1 network nodes
  - Estimated travel time: 0.00 seconds
  - Path start coords: (34.0674019, -118.2753526)



### Note -- baseline comparability
`DIST_MATRIX` from 3.3 = drop-in replacement for OR-Tools baseline too.
Feed same matrix to OR-Tools solver -> PPO vs OR-Tools compared on identical
real-road numbers, not toy euclidean. Needed for your Review 1 ablation table.